# Error Analysis, Debugging Models, and Comparing Models

Notebook for **Chapter 10** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch10.html

Tip: change a number, predict the output, then run the cell again.

## 10.1 Sanity Tests: Is the Pipeline Even Honest?

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
n = 4000
tenure = rng.integers(1, 60, n); spend = rng.normal(50, 15, n).round(1)
calls = rng.poisson(1.5, n)
plan = rng.choice(['basic','plus','pro'], n, p=[.5,.3,.2])
region = rng.choice(['N','S','E','W'], n)
z = -0.2 - 0.05*tenure + 0.03*(spend-50) + 0.3*calls + (plan=='basic')*0.6
z += ((tenure < 12) & (calls >= 3)) * 2.5          # the hidden segment
y = (rng.random(n) < 1/(1+np.exp(-z))).astype(int)
flip = rng.random(n) < 0.05                          # 5% label noise
y = np.where(flip, 1-y, y)
df = pd.DataFrame(dict(tenure=tenure, spend=spend, calls=calls,
                       plan=plan, region=region, churn=y))
X = pd.get_dummies(df.drop(columns='churn'), columns=['plan','region'], dtype=float)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

In [ ]:
d = DummyClassifier(strategy='prior').fit(Xtr, ytr)
print("dummy acc", round(accuracy_score(yte, d.predict(Xte)), 3))
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
gb = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
for nm, m in [('logreg', lr), ('gboost', gb)]:
    pr = m.predict_proba(Xte)[:, 1]
    print(nm, "acc", round(accuracy_score(yte, m.predict(Xte)), 3),
          "auc", round(roc_auc_score(yte, pr), 3), "logloss", round(log_loss(yte, pr), 3))

dummy acc 0.658
logreg acc 0.733 auc 0.752 logloss 0.549
gboost acc 0.722 auc 0.731 logloss 0.562


In [ ]:
ys = np.random.default_rng(1).permutation(ytr)
g2 = GradientBoostingClassifier(random_state=0).fit(Xtr, ys)
print("shuffled-label test auc", round(roc_auc_score(yte, g2.predict_proba(Xte)[:, 1]), 3))

idx = np.arange(40)
dt = DecisionTreeClassifier(random_state=0).fit(Xtr.iloc[idx], ytr[idx])
print("tiny-40 train acc, tree:", dt.score(Xtr.iloc[idx], ytr[idx]))

shuffled-label test auc 0.46
tiny-40 train acc, tree: 1.0


In [ ]:
rl = np.random.default_rng(5)
Xl = Xtr.copy(); Xl['days_since_cancel_request'] = np.where(ytr==1, rl.integers(0,5,len(ytr)), rl.integers(30,90,len(ytr)))
Xlt = Xte.copy(); Xlt['days_since_cancel_request'] = np.where(yte==1, rl.integers(0,5,len(yte)), rl.integers(30,90,len(yte)))
gl = GradientBoostingClassifier(random_state=0).fit(Xl, ytr)
print("auc", round(roc_auc_score(yte, gl.predict_proba(Xlt)[:, 1]), 3))
print(pd.Series(gl.feature_importances_, Xl.columns).sort_values(ascending=False).head(3).round(3))

auc 1.0
days_since_cancel_request    1.0
spend                        0.0
tenure                       0.0


## 10.2 Learning Curves and Validation Curves: Diagnosis by Picture

In [ ]:
from sklearn.model_selection import learning_curve, validation_curve, StratifiedKFold
cv5 = StratifiedKFold(5, shuffle=True, random_state=0)
sizes = [0.1, 0.25, 0.5, 0.75, 1.0]
s, tr, va = learning_curve(GradientBoostingClassifier(random_state=0), X, y,
                           train_sizes=sizes, cv=cv5, scoring='roc_auc')
for a, b, c in zip(s, tr.mean(1), va.mean(1)): print(a, round(b, 3), round(c, 3))
s2, tr2, va2 = learning_curve(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
                              X, y, train_sizes=sizes, cv=cv5, scoring='roc_auc')
for a, b, c in zip(s2, tr2.mean(1), va2.mean(1)): print("lr", a, round(b, 3), round(c, 3))

320 0.98 0.671
800 0.904 0.701
1600 0.841 0.713
2400 0.822 0.722
3200 0.802 0.727
lr 320 0.756 0.729
lr 800 0.743 0.733
lr 1600 0.738 0.738
lr 2400 0.747 0.74
lr 3200 0.745 0.74


In [ ]:
depths = [1, 2, 3, 5, 8]
tr, va = validation_curve(GradientBoostingClassifier(random_state=0, n_estimators=100), X, y,
                          param_name='max_depth', param_range=depths, cv=cv5, scoring='roc_auc')
for d_, a, b in zip(depths, tr.mean(1), va.mean(1)): print("depth", d_, round(a, 3), round(b, 3))

depth 1 0.748 0.733
depth 2 0.77 0.732
depth 3 0.802 0.727
depth 5 0.909 0.703
depth 8 0.999 0.676


## 10.3 Error Analysis: Read the Mistakes

In [ ]:
from sklearn.metrics import confusion_matrix
pr = lr.predict_proba(Xte)[:, 1]
print(confusion_matrix(yte, (pr >= 0.5).astype(int)))
te = df.loc[Xte.index].copy(); te['p'] = pr; te['y'] = yte
te['err'] = -(yte*np.log(pr) + (1-yte)*np.log(1-pr))
print(te.sort_values('err', ascending=False).head(5)[['tenure','spend','calls','plan','y','p','err']].round(2))
worst = te.sort_values('err', ascending=False).head(60)
print("worst60 tenure", worst.tenure.mean().round(1), "calls", worst.calls.mean().round(2),
      "| overall", te.tenure.mean().round(1), te.calls.mean().round(2))

[[709  81]
 [240 170]]
      tenure  spend  calls  plan  y     p   err
2269      54   26.8      0   plus  1  0.04  3.31
551       58   40.2      0   plus  1  0.05  3.00
785       41    4.8      0  basic  1  0.06  2.75
3075      54   29.3      2   plus  1  0.07  2.66
521       48   41.9      0   plus  1  0.08  2.47
worst60 tenure 43.4 calls 1.07 overall 30.4 1.52


In [ ]:
te['seg'] = np.where((te.tenure < 12) & (te.calls >= 3), 'new+calls>=3', 'other')
g = te.groupby('seg').apply(lambda g: pd.Series(dict(n=len(g), churn=g.y.mean(), pred=g.p.mean(),
        acc=((g.p >= .5) == g.y).mean(), logloss=g.err.mean())))
print(g.round(3))
te['tb'] = pd.cut(te.tenure, [0, 6, 12, 24, 60])
print(te.groupby('tb', observed=True).apply(lambda g: pd.Series(dict(n=len(g), churn=g.y.mean(),
        pred=g.p.mean(), acc=((g.p >= .5) == g.y).mean()))).round(3))

                  n  churn   pred    acc  logloss
seg                                               
new+calls>=3     41  0.927  0.709  0.854    0.392
other          1159  0.321  0.328  0.728    0.555
              n  churn   pred    acc
tb                                  
(0, 6]         115  0.530  0.583  0.643
(6, 12]        124  0.629  0.543  0.661
(12, 24]       234  0.397  0.442  0.679
(24, 60]       727  0.245  0.235  0.776


In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)
X2 = X.copy(); X2['new_calls'] = ((df.tenure < 12) & (df.calls >= 3)).astype(float)
mk = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
a = cross_val_score(mk(), X, y, cv=cv, scoring='roc_auc')
b = cross_val_score(mk(), X2, y, cv=cv, scoring='roc_auc')
print(round(a.mean(), 4), round(b.mean(), 4), round((b - a).mean(), 4))
l2 = mk().fit(X2.loc[Xtr.index], ytr)
p2 = l2.predict_proba(X2.loc[Xte.index])[:, 1]
print("segment mean prediction", p2[(te.seg == 'new+calls>=3').values].mean().round(3))

0.7414 0.7443 0.0029
segment mean prediction 0.921


## 10.4 Comparing Models Properly

In [ ]:
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
mods = {'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        'gboost': GradientBoostingClassifier(random_state=0),
        'forest': RandomForestClassifier(n_estimators=200, random_state=0, min_samples_leaf=5)}
sc = {k: cross_val_score(m, X, y, cv=cv, scoring='roc_auc') for k, m in mods.items()}
for k, v in sc.items(): print(k, v.mean().round(4), v.std().round(4))

logreg 0.7414 0.0164
gboost 0.728 0.0145
forest 0.7241 0.016


In [ ]:
d = sc['gboost'] - sc['logreg']
print("diff mean", d.mean().round(4), "sd", d.std(ddof=1).round(4))
print(stats.ttest_rel(sc['gboost'], sc['logreg']))
k = len(d); n_te = len(y)//5; n_tr = len(y) - n_te
var = d.var(ddof=1) * (1/k + n_te/n_tr)
t = d.mean()/np.sqrt(var)
print("corrected t", t.round(2), "p", (2*stats.t.sf(abs(t), k-1)).round(4))

diff mean -0.0134 sd 0.0101
TtestResult(statistic=np.float64(-4.206274330707449), pvalue=np.float64(0.0022853687483761443), df=np.int64(9))
corrected t -2.25 p 0.0511


In [ ]:
pa = gb.predict(Xte) == yte; pb = lr.predict(Xte) == yte
b = int((pa & ~pb).sum()); c = int((~pa & pb).sum())
stat = (abs(b - c) - 1)**2 / (b + c)
print("b", b, "c", c, "mcnemar", round(stat, 3), "p", round(stats.chi2.sf(stat, 1), 4))
rb = np.random.default_rng(0); ds = []
pg = gb.predict_proba(Xte)[:, 1]; pl = lr.predict_proba(Xte)[:, 1]
for _ in range(1000):
    i = rb.integers(0, len(yte), len(yte))
    ds.append(roc_auc_score(yte[i], pg[i]) - roc_auc_score(yte[i], pl[i]))
print("boot diff", np.mean(ds).round(4), np.percentile(ds, [2.5, 97.5]).round(4))

b 51 c 64
mcnemar 1.252 p 0.2631
boot diff -0.0216 [-0.034 -0.008]


## 10.5 Comparing Models Across Many Datasets: Wilcoxon, Friedman and Nemenyi

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
from scipy import stats
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_digits, make_classification, make_moons, make_circles
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
data = {"iris": load_iris(return_X_y=True), "wine": load_wine(return_X_y=True), "cancer": load_breast_cancer(return_X_y=True),
        "digits": load_digits(return_X_y=True), "moons": make_moons(600, noise=0.3, random_state=0),
        "circles": make_circles(600, noise=0.15, factor=0.5, random_state=0)}
for s in range(6):
    data["synth%d" % s] = make_classification(600, 12, n_informative=4 + s % 3, n_redundant=2, flip_y=0.05, class_sep=0.8, random_state=s)
models = {"logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
          "forest": RandomForestClassifier(100, random_state=0),
          "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(15))}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
names = list(data); S = np.array([[cross_val_score(m, *data[d], cv=cv).mean() for m in models.values()] for d in names])
print("%-9s" % "dataset", *["%7s" % k for k in models])
for d, row in zip(names, S): print("%-9s" % d, *["%7.3f" % v for v in row])
ranks = np.array([stats.rankdata(-row) for row in S]); R = ranks.mean(0)
print("mean rank (1 = best):", dict(zip(models, R.round(2))))
fr = stats.friedmanchisquare(*S.T); print("Friedman chi2 = %.2f, p = %.4f (N = %d datasets, k = 3 models)" % (fr.statistic, fr.pvalue, len(names)))
N, k = len(names), 3; cd = 2.343 * np.sqrt(k * (k + 1) / (6 * N))
print("Nemenyi critical difference (alpha 0.05, q = 2.343): %.2f" % cd)
print("rank gaps: logreg-forest %.2f, knn-forest %.2f, logreg-knn %.2f" % (R[0] - R[1], R[2] - R[1], R[0] - R[2]))
for a, b in [(0, 1), (1, 2), (0, 2)]:
    w = stats.wilcoxon(S[:, a], S[:, b]); print("Wilcoxon %s vs %s: W = %.0f, p = %.4f, median diff %.4f" % (list(models)[a], list(models)[b], w.statistic, w.pvalue, np.median(S[:, a] - S[:, b])))

dataset    logreg  forest     knn
iris        0.960   0.940   0.953
wine        0.983   0.972   0.967
cancer      0.979   0.965   0.961
digits      0.969   0.974   0.962
moons       0.863   0.902   0.900
circles     0.472   0.937   0.960
synth0      0.663   0.923   0.838
synth1      0.718   0.842   0.780
synth2      0.718   0.822   0.787
synth3      0.768   0.853   0.807
synth4      0.862   0.893   0.872
synth5      0.810   0.877   0.857
mean rank (1 = best): {'logreg': np.float64(2.42), 'forest': np.float64(1.42), 'knn': np.float64(2.17)}
Friedman chi2 = 6.50, p = 0.0388 (N = 12 datasets, k = 3 models)
Nemenyi critical difference (alpha 0.05, q = 2.343): 0.96
rank gaps: logreg-forest 1.00, knn-forest 0.75, logreg-knn 0.25
Wilcoxon logreg vs forest: W = 9, p = 0.0161, median diff -0.0525
Wilcoxon forest vs knn: W = 13, p = 0.0425, median diff 0.0158
Wilcoxon logreg vs knn: W = 12, p = 0.0342, median diff -0.0375


## Chapter recap

## Practice set

Problem 10.P.1 (easy) 1,000 emails, 950 are not spam. What accuracy does "always not spam" get?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch10.html